# Mining Equipment Condition Monitoring Capstone

Saskatchewan Polytechnic PROJ 613, Group 9 (Ganesh Mishra, Lakshya Sharma, Sagar Patel). **Week 4 checkpoint.**

The professor asked us to compare **sensor-based** and **OpenCV/image-based** predictions for a genuinely comparable equipment component and fault. This notebook reruns the project's own scripts from start to finish and shows the two completed comparisons:

| # | Equipment category | Component and target | Sensor branch | Visual branch |
|---|---|---|---|---|
| 1 | Pump | Support bearing: healthy, inner race, outer race, ball fault | University of Ottawa vibration (bearing test rig) | XJTU event camera, OpenCV HOG |
| 2 | Belt Conveyor | Geared drive reducer: healthy vs gear-worn | CORA T1 temperature | CORA infrared, OpenCV |

Six further categories (Haul/Dump Truck, Excavator, Dozer, Wheel Loader, Drill Rig, Grader) are **mapped but not modelled**; see `Dataset_Manifest/manifest.md`.

**What this notebook does not claim:** validation on real mining machines, sensorâ€“image fusion, paired sensor/image samples, or prediction of a future repair date. The models classify **present** component condition on laboratory rigs.

The notebook calls the scripts in `Preprocessing/` and `Models/` instead of copying their code, so it always matches them.

## Reproducibility record

The cell below records the exact environment. Package versions matter here: one supporting baseline gives a different test split under a different scikit-learn version (Section 5).

In [1]:
import contextlib
import io
import json
import os
import platform
import runpy
import subprocess
import sys
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

here = Path.cwd().resolve()
CAPSTONE = next(p for p in (here, *here.parents) if (p / "Models").is_dir() and (p / "Preprocessing").is_dir())
RAW = CAPSTONE / "Raw_Data"
PROCESSED = CAPSTONE / "Processed_Data"
RESULTS = CAPSTONE / "Results"

def package(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"

try:
    commit = subprocess.check_output(["git", "-C", str(CAPSTONE), "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    commit = "unknown"

record = {"git commit": commit, "python": platform.python_version(), "platform": platform.platform()}
record.update({name: package(name) for name in
               ("numpy", "scipy", "pandas", "scikit-learn", "opencv-python", "matplotlib", "remotezip", "openpyxl")})
display(pd.Series(record, name="value").to_frame())

,value
git commit,a70bb6d
python,3.11.9
platform,Windows-10-10.0.26100-SP0
numpy,2.4.6
scipy,1.17.1
pandas,2.3.3
scikit-learn,1.9.1
opencv-python,4.14.0.94
matplotlib,3.11.2
remotezip,0.12.6


## Data sources and local availability

Raw data never goes into Git (`Raw_Data/` and `Processed_Data/` are ignored). Each source has a download or intake route documented in the folder `README.md`. All sources are CC BY 4.0 except where noted in the manifest.

In [2]:
def count(folder, pattern):
    path = RAW / folder
    files = [p for p in path.rglob(pattern) if p.is_file()] if path.is_dir() else []
    return len(files), sum(p.stat().st_size for p in files) / 1e6

sources = [
    ("1 Pump", "Vibration: University of Ottawa V2 (DOI 10.17632/v43hmbwxpm.2)", "uottawa_bearing", "*.mat", 48),
    ("1 Pump", "Event camera: XJTU-DV-Pump, Zenodo 17447083", "xjtu_dv_pump", "*.zip", 2),
    ("2 Belt Conveyor", "Infrared: CORA v2.1 (DOI 10.34810/DATA2500)", "cora_gear_wear", "*.mat", 16),
    ("5 Supporting", "UCI hydraulic system", "uci_hydraulic", "*.zip", 1),
    ("5 Supporting", "ESPset V3 features", "espset_v3", "*.zip", 1),
    ("5 Supporting", "PHM 2022 rock drill", "phm2022_rock_drill", "*.zip", 1),
    ("5 Supporting", "Original centrifugal-pump XLSX (diagnostic)", "pump_bearing", "*.xlsx", 4),
]
rows = []
for section, label, folder, pattern, expected in sources:
    n, mb = count(folder, pattern)
    rows.append({"section": section, "source": label, "files": f"{n}/{expected}", "MB": round(mb, 1), "local": n >= expected})
t1 = PROCESSED / "cora_gear_wear" / "t1_bounded_samples.npz"
rows.insert(3, {"section": "2 Belt Conveyor", "source": "T1 temperature samples: CORA v2.1 (bounded byte ranges)",
                "files": f"{int(t1.is_file())}/1", "MB": round(t1.stat().st_size / 1e6, 1) if t1.is_file() else 0.0,
                "local": t1.is_file()})
availability = pd.DataFrame(rows)
display(availability)

EVENT_ZIPS_LOCAL = (RAW / "xjtu_dv_pump" / "XJTU-DV-Pump_part1.zip").is_file() and (RAW / "xjtu_dv_pump" / "XJTU-DV-Pump_part2.zip").is_file()
ALL_LOCAL = bool(availability["local"].all())
print("All data is local: running fully offline." if ALL_LOCAL else
      "Some data is not local: those steps are skipped, and the event-camera branch reads Zenodo over the internet.")

,section,source,files,MB,local
0,1 Pump,Vibration: University of Ottawa V2 (DOI 10.176...,48/48,631.5,True
1,1 Pump,"Event camera: XJTU-DV-Pump, Zenodo 17447083",2/2,16488.7,True
2,2 Belt Conveyor,Infrared: CORA v2.1 (DOI 10.34810/DATA2500),16/16,390.5,True
3,2 Belt Conveyor,T1 temperature samples: CORA v2.1 (bounded byt...,1/1,1.7,True
4,5 Supporting,UCI hydraulic system,1/1,76.6,True
5,5 Supporting,ESPset V3 features,1/1,0.3,True
6,5 Supporting,PHM 2022 rock drill,1/1,235.4,True
7,5 Supporting,Original centrifugal-pump XLSX (diagnostic),4/4,3.1,True


All data is local: running fully offline.


In [3]:
import zipfile
import remotezip

if ALL_LOCAL:
    # Offline guard: any network call now fails loudly, which proves the run is local.
    for name in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):
        os.environ[name] = "http://127.0.0.1:9"
    os.environ["NO_PROXY"] = ""

if EVENT_ZIPS_LOCAL:
    def local_zip(url, *args, **kwargs):
        part = "part1" if "part1" in url else "part2" if "part2" in url else None
        if part is None:
            raise ValueError(f"Unexpected archive URL: {url}")
        return zipfile.ZipFile(RAW / "xjtu_dv_pump" / f"XJTU-DV-Pump_{part}.zip")
    remotezip.RemoteZip = local_zip  # the event-camera script reads the same members from the local zips


def run_script(relative_path, *args, tail=3):
    """Run one project script exactly as `python <script>` would; show only its last lines."""
    path = CAPSTONE / relative_path
    buffer, saved = io.StringIO(), sys.argv
    sys.argv = [path.name, *args]
    try:
        with contextlib.redirect_stdout(buffer):
            runpy.run_path(str(path), run_name="__main__")
    finally:
        sys.argv = saved
    text = buffer.getvalue().replace(str(CAPSTONE) + os.sep, "").replace(str(CAPSTONE), ".")
    lines = [line for line in text.splitlines() if line.strip()]
    print(f"[{relative_path}]")
    for line in lines[-tail:]:
        print("  " + line)


def committed(json_name):
    path = RESULTS / json_name
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else None


def metrics(block):
    keys = ("accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1")
    return {k: round(block[k], 3) for k in keys if isinstance(block.get(k), (int, float))}


def reproduced(before, json_name, paths):
    after = json.loads((RESULTS / json_name).read_text(encoding="utf-8"))
    def pick(doc, path):
        for key in path:
            doc = doc[key]
        return metrics(doc)
    same = before is not None and all(pick(before, p) == pick(after, p) for p in paths)
    print(f"{json_name}: {'matches' if same else 'DIFFERS from'} the committed result")
    return after

## 1. Pump support bearing: vibration vs event camera

**Sensor branch:** University of Ottawa accelerometer recordings from a bearing **test rig**, used as a pump support-bearing proxy. Each 10 s recording becomes 100 windows with 16 time/spectral features; a random forest trains on the 24 A/B speed-profile recordings and predicts each of the 24 separate C/D recordings once.

**Visual branch:** XJTU event-camera recordings of a **laboratory pump** bearing housing. The first two seconds of each recording become eight 250 ms event-count images; OpenCV HOG features feed a random forest trained on 20 Hz recordings and tested on the 24 separate 30 Hz recordings. These are motion images, not photographs of visible damage.

Both branches hold out whole recordings, so no window or frame of a test recording is seen in training.

In [4]:
before_sensor = committed("pump_bearing_independent.json")
run_script("Models/pump_bearing_independent_baseline.py")
pump_sensor = reproduced(before_sensor, "pump_bearing_independent.json",
                         [("random_forest_run_level",), ("held_out_speed_profile_run_level",), ("dummy_run_level",)])

before_visual = committed("pump_event_visual.json")
run_script("Models/pump_event_visual_baseline.py")
pump_visual = reproduced(before_visual, "pump_event_visual.json",
                         [("random_forest_run_level",), ("held_out_camera_configuration_run_level",), ("dummy_run_level",)])

run_script("Models/pump_compare_modalities.py", tail=2)

[Models/pump_bearing_independent_baseline.py]
  Saved Results\pump_bearing_independent.json and Figures\pump_bearing_independent_confusions.png
  Held-out run accuracy 0.9583; macro F1 0.9580
pump_bearing_independent.json: matches the committed result


[Models/pump_event_visual_baseline.py]
  Saved Results\pump_event_visual.json and Figures\pump_event_visual_confusion.png
  Run-level test macro F1: 1.0000
pump_event_visual.json: matches the committed result


[Models/pump_compare_modalities.py]
  Saved Results\pump_sensor_visual_comparison.md and Figures\pump_sensor_visual_confusions.png


In [5]:
def first_metrics(doc, candidates):
    for path in candidates:
        node = doc
        try:
            for key in path:
                node = node[key]
        except (KeyError, TypeError):
            continue
        found = metrics(node)
        if found:
            return found
    return {}

pump_rows = {
    "Vibration: held-out C/D recordings (24)": first_metrics(pump_sensor, [("random_forest_run_level",)]),
    "Event camera: held-out 30 Hz recordings (24)": metrics(pump_visual["random_forest_run_level"]),
    "Event camera: held-out camera configurations (48)": metrics(pump_visual["held_out_camera_configuration_run_level"]),
    "Most-frequent-class control (24)": metrics(pump_visual["dummy_run_level"]),
}
display(pd.DataFrame(pump_rows).T)

,accuracy,macro_precision,macro_recall,macro_f1
Vibration: held-out C/D recordings (24),0.958,0.964,0.958,0.958
Event camera: held-out 30 Hz recordings (24),1.000,1.000,1.000,1.000
Event camera: held-out camera configurations (48),0.854,0.868,0.854,0.855
Most-frequent-class control (24),0.250,0.062,0.250,0.100


![Pump confusion matrices](../Figures/pump_sensor_visual_confusions.png)

**Interpretation.** Both branches separate the four bearing conditions well on held-out recordings (vibration 0.958, event camera 1.000), and both are far above the 0.250 most-frequent control. The harder event-camera check, which holds out whole camera configurations, drops to about 0.85. The branches use **different rigs and operating shifts**, so the gap between them is not evidence that one modality is better. This is a component-level proof of concept, not a validated mining-pump result. Full write-up: `Results/pump_sensor_visual_comparison.md`.

## 2. Belt Conveyor geared drive: T1 temperature vs infrared

**Source:** CORA v2.1, one controlled motor/gearbox **laboratory rig**, used as a component analogue for a geared conveyor drive reducer. It says nothing about belt cracks, idlers or gearless drives.

**Target:** healthy (`H`) vs gear-worn (`W25`, `W50`, `W75`), 16 stationary runs at four speeds (F5, F15, F50, F60). Each speed is held out in turn; one prediction per run. The sensor branch uses eight bounded windows of each T1 CSV (about 4.3% of its bytes), and the visual branch uses all 450 infrared frames per run.

In [6]:
before_belt = committed("belt_conveyor_cora.json")
run_script("Models/belt_conveyor_cora_baseline.py")
belt = reproduced(before_belt, "belt_conveyor_cora.json",
                  [("sensor", "model"), ("infrared", "model"), ("sensor", "most_frequent_control")])

belt_rows = {
    "T1 temperature sensor (16 runs)": metrics(belt["sensor"]["model"]),
    "Infrared / OpenCV (16 runs)": metrics(belt["infrared"]["model"]),
    "Most-frequent-class control (16 runs)": metrics(belt["sensor"]["most_frequent_control"]),
}
display(pd.DataFrame(belt_rows).T)

[Models/belt_conveyor_cora_baseline.py]
  Extracted run 15/16: W75_F50_S
  Extracted run 16/16: W75_F60_S
  Saved Results\belt_conveyor_cora.json, Results\belt_conveyor_sensor_visual_comparison.md, and Figures\belt_conveyor_cora_confusions.png
belt_conveyor_cora.json: matches the committed result


,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
T1 temperature sensor (16 runs),0.625,0.583,0.567,0.583,0.564
Infrared / OpenCV (16 runs),0.500,0.417,0.433,0.417,0.418
Most-frequent-class control (16 runs),0.750,0.500,0.375,0.500,0.429


![Belt Conveyor confusion matrices](../Figures/belt_conveyor_cora_confusions.png)

**Interpretation.** Both learnt models score **below the 0.750 majority-class accuracy**, and infrared catches only one of four healthy runs. With 16 runs from one rig and one run per wear state and speed, this is a completed, leakage-aware comparison with **weak generalization**, not an effective fault detector. Full write-up: `Results/belt_conveyor_sensor_visual_comparison.md`.

## 3. Side-by-side summary

In [7]:
summary = pd.DataFrame([
    {"equipment": "Pump", "branch": "Vibration sensor", **first_metrics(pump_sensor, [("random_forest_run_level",)])},
    {"equipment": "Pump", "branch": "Event camera (OpenCV)", **metrics(pump_visual["random_forest_run_level"])},
    {"equipment": "Belt Conveyor", "branch": "T1 temperature sensor", **metrics(belt["sensor"]["model"])},
    {"equipment": "Belt Conveyor", "branch": "Infrared (OpenCV)", **metrics(belt["infrared"]["model"])},
]).set_index(["equipment", "branch"])[["accuracy", "macro_precision", "macro_recall", "macro_f1"]]
display(summary)

accuracy  macro_precision  macro_recall  \
equipment     branch                                                           
Pump          Vibration sensor          0.958            0.964         0.958   
              Event camera (OpenCV)     1.000            1.000         1.000   
Belt Conveyor T1 temperature sensor     0.625            0.567         0.583   
              Infrared (OpenCV)         0.500            0.433         0.417   

                                     macro_f1  
equipment     branch                           
Pump          Vibration sensor          0.958  
              Event camera (OpenCV)     1.000  
Belt Conveyor T1 temperature sensor     0.564  
              Infrared (OpenCV)         0.418

Scores are comparable **within** an equipment row pair only as descriptions of each fixed pipeline on its own held-out units. They are not comparable across equipment, and they do not rank sensors against cameras in general.

## 4. Supporting sensor-only baselines

These are **sensor-only** benchmarks. They do **not** count as sensor-versus-image experiments for any equipment category, and their scores are not comparable with each other.

**Known reproducibility issue.** The committed results were produced with package versions that were not recorded. Rerunning them in a different environment changes some numbers:

| Baseline | Committed test macro F1 | scikit-learn 1.9.1 | scikit-learn 1.6.1 |
|---|---:|---:|---:|
| UCI hydraulic | 0.970 (299 test cycles) | 0.963 (290 test cycles) | 0.970, same test split |
| ESPset V3 | 0.918 | 0.914 | 0.919 |
| PHM 2022 rock drill | 0.776 | 0.798 | 0.797 |

The hydraulic split uses `StratifiedGroupKFold(shuffle=True)`, which assigns groups differently across scikit-learn versions. The rock-drill difference remains under both versions and is not yet explained. The reproducibility record at the top of this notebook now records versions so the issue cannot recur silently.

In [8]:
supporting = [
    ("Preprocessing/hydraulic_baseline.py", "uci_hydraulic_pump_leakage.json", "UCI hydraulic: pump leakage severity (3 classes)"),
    ("Models/espset_baseline.py", "espset_v3_binary_fault.json", "ESPset V3: normal vs abnormal, pump IDs held out"),
    ("Models/rock_drill_baseline.py", "phm2022_rock_drill.json", "PHM 2022 rock drill: 11 conditions, configurations held out"),
]
support_rows = {}
for script, json_name, label in supporting:
    before = committed(json_name)
    try:
        run_script(script, tail=1)
    except (FileNotFoundError, ValueError) as error:
        print(f"Skipped {script}: {error}")
        continue
    after = json.loads((RESULTS / json_name).read_text(encoding="utf-8"))
    support_rows[label] = {"committed macro F1": round(before["test_selected"]["macro_f1"], 3) if before else None,
                           "this run macro F1": round(after["test_selected"]["macro_f1"], 3),
                           "this run accuracy": round(after["test_selected"]["accuracy"], 3)}
display(pd.DataFrame(support_rows).T)

[Preprocessing/hydraulic_baseline.py]
  Wrote Results\uci_hydraulic_pump_leakage.json and Figures\uci_hydraulic_pump_leakage.png


[Models/espset_baseline.py]
  Wrote Results\espset_v3_binary_fault.json and Figures\espset_v3_binary_fault.png


[Models/rock_drill_baseline.py]
  Wrote Results\phm2022_rock_drill.json and Figures\phm2022_rock_drill.png


,committed macro F1,this run macro F1,this run accuracy
UCI hydraulic: pump leakage severity (3 classes),0.963,0.963,0.962
"ESPset V3: normal vs abnormal, pump IDs held out",0.914,0.914,0.937
"PHM 2022 rock drill: 11 conditions, configurations held out",0.798,0.798,0.813


## 5. Conclusions and next steps

- **Pump** is the stronger matched-condition demonstration: both modalities classify the same four present bearing conditions on held-out recordings.
- **Belt Conveyor** completes a same-rig sensor/infrared comparison, but both models are below the majority-class control.
- The remaining six equipment categories are planned, not implemented. Five of them reuse the same CORA laboratory analogue, which does not create independent validations.
- **Next:** review the scope and the component-transfer assumptions with the professor; if accepted, pre-register one remaining category (classes, whole-recording test units, metrics and claims fixed in advance) before modelling it.

**Data citations:** University of Ottawa bearing vibration V2 (DOI 10.17632/v43hmbwxpm.2); XJTU-DV (Zenodo 10.5281/zenodo.17447083); CORA v2.1 (DOI 10.34810/DATA2500); all CC BY 4.0. Supporting datasets are cited in `Dataset_Manifest/manifest.md`.